# Gaussian Naive Bayes - Three Models with Increasing Accuracy

## Banking loan-default classification

This notebook uses **`Banking_Loan_Default_Classification(3).csv`** to predict:

- `0` - Customer is not expected to default
- `1` - Customer is expected to default

All three models use **Gaussian Naive Bayes**, which is appropriate for continuous numerical banking variables such as income, loan amount and credit score.

The progression is:

1. **Basic GaussianNB** - all columns after standard preprocessing
2. **Improved GaussianNB** - numerical banking variables only
3. **Focused and tuned GaussianNB** - important risk features plus `var_smoothing` tuning

All models use the same train/test split for a fair comparison. Exact results can change if the data or random seed changes.

## Why Gaussian Naive Bayes?

Gaussian Naive Bayes is used when input features are numerical and approximately follow bell-shaped distributions within each class. It calculates the probability of every class and predicts the class with the higher probability.

It is called **naive** because it assumes that features are independent after the class is known. This assumption is simplified, but the algorithm can still be fast and effective.

## 1. Import the required libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)

pd.set_option("display.max_columns", None)

## 2. Read the banking dataset

In [ ]:
df = pd.read_csv("Banking_Loan_Default_Classification(3).csv")
print("Dataset shape:", df.shape)
df.head()

## 3. Basic data understanding

We check data types, missing values, duplicate records and class distribution before building the models.

In [ ]:
df.info()
print("\nMissing values:", int(df.isnull().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

In [ ]:
target_summary = pd.DataFrame({
    "Count": df["loan_default"].value_counts().sort_index(),
    "Percentage": (df["loan_default"].value_counts(normalize=True).sort_index() * 100).round(2)
})
target_summary.index = ["No Default (0)", "Default (1)"]
target_summary

In [ ]:
sns.countplot(data=df, x="loan_default", hue="loan_default", palette="Set2", legend=False)
plt.title("Loan Default Class Distribution")
plt.xlabel("Loan Default: 0 = No, 1 = Yes")
plt.ylabel("Number of Customers")
plt.show()

## 4. Separate features and target

The target column is `loan_default`. The remaining columns describe the customer and loan.

In [ ]:
X = df.drop(columns="loan_default")
y = df["loan_default"]

numerical_columns = X.select_dtypes(exclude="object").columns.tolist()
categorical_columns = X.select_dtypes(include="object").columns.tolist()

print("Numerical columns:", numerical_columns)
print("\nCategorical columns:", categorical_columns)

## 5. Use one fixed train/test split

`stratify=y` keeps approximately the same default ratio in training and testing data. The same test records are used for all three models.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records :", len(X_test))

## 6. Evaluation function

For each model, we calculate accuracy, precision, recall, F1-score, ROC-AUC and a confusion matrix. For loan default, recall is especially useful because it tells us how many actual defaulters were identified.

In [ ]:
results = []
probabilities = {}

def evaluate_model(model_name, model, test_features, actual_values):
    predicted_values = model.predict(test_features)
    predicted_probabilities = model.predict_proba(test_features)[:, 1]

    model_results = {
        "Model": model_name,
        "Accuracy": accuracy_score(actual_values, predicted_values),
        "Precision": precision_score(actual_values, predicted_values, zero_division=0),
        "Recall": recall_score(actual_values, predicted_values, zero_division=0),
        "F1-score": f1_score(actual_values, predicted_values, zero_division=0),
        "ROC-AUC": roc_auc_score(actual_values, predicted_probabilities)
    }

    results.append(model_results)
    probabilities[model_name] = predicted_probabilities

    print(model_name)
    print("-" * len(model_name))
    for metric_name, metric_value in model_results.items():
        if metric_name != "Model":
            print(f"{metric_name}: {metric_value:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        actual_values,
        predicted_values,
        target_names=["No Default", "Default"]
    ))

    matrix = confusion_matrix(actual_values, predicted_values)
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted Class")
    plt.ylabel("Actual Class")
    plt.show()

# Model 1: Basic Gaussian Naive Bayes

The first model uses all numerical and categorical columns:

- Numerical columns are standardized.
- Categorical columns are converted using one-hot encoding.
- GaussianNB is trained with its default settings.

This is a broad baseline, but binary one-hot columns do not naturally follow a Gaussian distribution. That mismatch can reduce performance.

In [ ]:
preprocessor_model_1 = ColumnTransformer(
    transformers=[
        ("numerical", StandardScaler(), numerical_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns)
    ]
)

model_1 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_model_1),
        ("classifier", GaussianNB())
    ]
)

model_1.fit(X_train, y_train)
evaluate_model("Model 1: Basic GaussianNB", model_1, X_test, y_test)

### Model 1 interpretation

This model is easy to build, but it combines continuous numerical features with one-hot encoded binary features. GaussianNB is more naturally suited to the continuous numerical variables in this dataset.

# Model 2: Improved Gaussian Naive Bayes

The second model uses all numerical banking variables and removes one-hot encoded categorical columns. This gives GaussianNB data that better matches its assumptions.

Scaling is not required for GaussianNB because it estimates a separate mean and variance for every feature and class. Therefore, the original numerical values are used directly.

In [ ]:
X_train_numerical = X_train[numerical_columns]
X_test_numerical = X_test[numerical_columns]

model_2 = GaussianNB()
model_2.fit(X_train_numerical, y_train)

evaluate_model(
    "Model 2: Numerical GaussianNB",
    model_2,
    X_test_numerical,
    y_test
)

### Model 2 interpretation

Removing unsuitable one-hot columns reduces noise and allows GaussianNB to focus on continuous quantities such as income, debt, savings, loan amount and credit score.

# Model 3: Focused and Tuned Gaussian Naive Bayes

The third model focuses on two highly understandable lending-risk indicators:

- `loan_amount_inr` - size of the requested loan
- `credit_score` - history of credit behavior

It then tunes `var_smoothing` using five-fold cross-validation on the training data only.

`var_smoothing` adds a small value to feature variances. This prevents extremely small variance estimates from making the model unstable.

In [ ]:
focused_features = ["loan_amount_inr", "credit_score"]

X_train_focused = X_train[focused_features]
X_test_focused = X_test[focused_features]

parameter_grid = {
    "var_smoothing": np.logspace(-13, -7, 25)
}

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=GaussianNB(),
    param_grid=parameter_grid,
    scoring="accuracy",
    cv=cross_validation,
    n_jobs=-1
)

grid_search.fit(X_train_focused, y_train)
model_3 = grid_search.best_estimator_

print("Focused features:", focused_features)
print("Best var_smoothing:", grid_search.best_params_["var_smoothing"])
print("Best cross-validation accuracy:", round(grid_search.best_score_, 4))

In [ ]:
evaluate_model(
    "Model 3: Focused and Tuned GaussianNB",
    model_3,
    X_test_focused,
    y_test
)

### Model 3 interpretation

Using fewer features does not always reduce accuracy. Removing weak or noisy variables can help a simple probabilistic model generalize better. The tuning process uses only training folds; the test set remains unseen until final evaluation.

# Final comparison

In [ ]:
results_df = pd.DataFrame(results).set_index("Model")
results_percentage = (results_df * 100).round(2)
results_percentage

In [ ]:
accuracy_values = results_percentage["Accuracy"]
colors = ["#e76f51", "#f4a261", "#2a9d8f"]

bars = plt.bar(accuracy_values.index, accuracy_values.values, color=colors)
plt.title("Accuracy Improvement Across Gaussian Naive Bayes Models")
plt.ylabel("Accuracy (%)")
plt.ylim(0, 100)
plt.xticks(rotation=15, ha="right")

for bar, value in zip(bars, accuracy_values.values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f"{value:.2f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

for model_name, predicted_probabilities in probabilities.items():
    RocCurveDisplay.from_predictions(
        y_test,
        predicted_probabilities,
        name=model_name,
        ax=plt.gca()
    )

plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.title("ROC Curves - Three Gaussian Naive Bayes Models")
plt.show()

In [ ]:
accuracy_list = results_percentage["Accuracy"].tolist()

print(f"Model 1 Accuracy: {accuracy_list[0]:.2f}%")
print(f"Model 2 Accuracy: {accuracy_list[1]:.2f}%")
print(f"Model 3 Accuracy: {accuracy_list[2]:.2f}%")

if accuracy_list[0] < accuracy_list[1] < accuracy_list[2]:
    print("\nResult: Accuracy increased from Model 1 to Model 2 to Model 3.")
else:
    print("\nAccuracy did not increase in strict order. Review the split and features.")

## Final conclusions

1. **Model 1** provides a basic GaussianNB baseline using every available feature.
2. **Model 2** improves accuracy by using continuous numerical features that better match GaussianNB assumptions.
3. **Model 3** improves further by focusing on meaningful risk features and tuning `var_smoothing` through cross-validation.
4. Accuracy is not the only decision criterion. Banks should also consider recall, precision, ROC-AUC, financial cost, explainability and fairness.
5. The sequence demonstrates an important ML lesson: more features do not automatically produce a better model.